# Healthcare Claims Fraud Detection
## Unsupervised Anomaly Detection using Isolation Forest
---

In [ ]:
%%sql -r claims_created
CREATE OR REPLACE TABLE HEALTHCARE.ENROLLMENT.CLAIMS_DETAIL AS
WITH date_range AS (
    SELECT 
        MEMBER_ID,
        MEMBER_NAME,
        STATE,
        AGE,
        PRODUCT_CODE,
        ENROLLMENT_STATUS,
        CLAIMS_COUNT_LAST_YEAR,
        TOTAL_CLAIMS_AMOUNT,
        ENROLLMENT_START_DATE,
        ENROLLMENT_END_DATE
    FROM HEALTHCARE.ENROLLMENT.MEMBER_ENROLLMENT
    WHERE CLAIMS_COUNT_LAST_YEAR > 0
),
claim_generator AS (
    SELECT 
        dr.*,
        ROW_NUMBER() OVER (PARTITION BY dr.MEMBER_ID ORDER BY RANDOM()) AS claim_seq
    FROM date_range dr,
        LATERAL FLATTEN(ARRAY_GENERATE_RANGE(0, dr.CLAIMS_COUNT_LAST_YEAR)) f
)
SELECT 
    'CLM-' || LPAD(ROW_NUMBER() OVER (ORDER BY RANDOM())::VARCHAR, 7, '0') AS CLAIM_ID,
    MEMBER_ID,
    MEMBER_NAME,
    STATE,
    AGE,
    PRODUCT_CODE,
    ENROLLMENT_STATUS,
    DATEADD('day', UNIFORM(1, 365, RANDOM()), '2024-01-01'::DATE) AS CLAIM_DATE,
    CASE 
        WHEN UNIFORM(1, 100, RANDOM()) <= 3 THEN ROUND(UNIFORM(15000, 95000, RANDOM())::FLOAT, 2)
        WHEN UNIFORM(1, 100, RANDOM()) <= 15 THEN ROUND(UNIFORM(5000, 15000, RANDOM())::FLOAT, 2)
        ELSE ROUND(UNIFORM(50, 5000, RANDOM())::FLOAT, 2)
    END AS CLAIM_AMOUNT,
    CASE 
        WHEN UNIFORM(1,100,RANDOM()) <= 40 THEN 'IN_NETWORK'
        WHEN UNIFORM(1,100,RANDOM()) <= 70 THEN 'OUT_OF_NETWORK'
        ELSE 'EMERGENCY'
    END AS NETWORK_TYPE,
    'PRV-' || LPAD(UNIFORM(1, 200, RANDOM())::VARCHAR, 4, '0') AS PROVIDER_ID,
    CASE
        WHEN PRODUCT_CODE = 'MEDICAL' THEN 
            CASE UNIFORM(1,5,RANDOM())
                WHEN 1 THEN 'CONSULTATION'
                WHEN 2 THEN 'SURGERY'
                WHEN 3 THEN 'DIAGNOSTIC'
                WHEN 4 THEN 'THERAPY'
                ELSE 'PRESCRIPTION'
            END
        WHEN PRODUCT_CODE = 'DENTAL' THEN
            CASE UNIFORM(1,4,RANDOM())
                WHEN 1 THEN 'CLEANING'
                WHEN 2 THEN 'FILLING'
                WHEN 3 THEN 'ROOT_CANAL'
                ELSE 'EXTRACTION'
            END
        ELSE
            CASE UNIFORM(1,3,RANDOM())
                WHEN 1 THEN 'GENERIC_DRUG'
                WHEN 2 THEN 'BRAND_DRUG'
                ELSE 'SPECIALTY_DRUG'
            END
    END AS SERVICE_TYPE,
    CASE UNIFORM(1,4,RANDOM())
        WHEN 1 THEN 'APPROVED'
        WHEN 2 THEN 'APPROVED'
        WHEN 3 THEN 'APPROVED'
        ELSE 'DENIED'
    END AS CLAIM_STATUS,
    CASE 
        WHEN DAYOFWEEK(DATEADD('day', UNIFORM(1, 365, RANDOM()), '2024-01-01'::DATE)) IN (0, 6) THEN 1
        ELSE 0
    END AS IS_WEEKEND_CLAIM
FROM claim_generator

In [ ]:
%%sql -r claims_overview
SELECT COUNT(*) AS total_claims, 
       COUNT(DISTINCT MEMBER_ID) AS unique_members,
       ROUND(AVG(CLAIM_AMOUNT), 2) AS avg_claim,
       ROUND(MAX(CLAIM_AMOUNT), 2) AS max_claim
FROM HEALTHCARE.ENROLLMENT.CLAIMS_DETAIL

## 1. Load Data & Basic EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = claims_overview.copy()
print(f"Quick Overview:")
print(df)
print("\n--- Loading full claims data ---")
from snowflake.snowpark.context import get_active_session
session = get_active_session()
df = session.sql('SELECT * FROM HEALTHCARE.ENROLLMENT.CLAIMS_DETAIL').to_pandas()
df['CLAIM_DATE'] = pd.to_datetime(df['CLAIM_DATE'])

print(f"\nShape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nBasic Stats:")
print(f"  Total Claims: {len(df):,}")
print(f"  Unique Members: {df['MEMBER_ID'].nunique():,}")
print(f"  Date Range: {df['CLAIM_DATE'].min().date()} to {df['CLAIM_DATE'].max().date()}")
print(f"  Avg Claim Amount: ${df['CLAIM_AMOUNT'].mean():,.2f}")
print(f"  Median Claim Amount: ${df['CLAIM_AMOUNT'].median():,.2f}")
print(f"  Max Claim Amount: ${df['CLAIM_AMOUNT'].max():,.2f}")
print(f"\nClaim Status Distribution:")
print(df['CLAIM_STATUS'].value_counts())
print(f"\nService Type Distribution:")
print(df['SERVICE_TYPE'].value_counts())

## 2. Feature Engineering for Fraud Detection

In [ ]:
member_features = df.groupby('MEMBER_ID').agg(
    total_claims=('CLAIM_ID', 'count'),
    total_amount=('CLAIM_AMOUNT', 'sum'),
    avg_claim_amount=('CLAIM_AMOUNT', 'mean'),
    max_claim_amount=('CLAIM_AMOUNT', 'max'),
    std_claim_amount=('CLAIM_AMOUNT', 'std'),
    unique_providers=('PROVIDER_ID', 'nunique'),
    unique_service_types=('SERVICE_TYPE', 'nunique'),
    weekend_claims=('IS_WEEKEND_CLAIM', 'sum'),
    denied_claims=('CLAIM_STATUS', lambda x: (x == 'DENIED').sum()),
    out_of_network=('NETWORK_TYPE', lambda x: (x == 'OUT_OF_NETWORK').sum()),
).reset_index()

member_features['claim_frequency_per_month'] = member_features['total_claims'] / 12
member_features['weekend_ratio'] = member_features['weekend_claims'] / member_features['total_claims']
member_features['denied_ratio'] = member_features['denied_claims'] / member_features['total_claims']
member_features['out_of_network_ratio'] = member_features['out_of_network'] / member_features['total_claims']
member_features['amount_per_provider'] = member_features['total_amount'] / member_features['unique_providers']
member_features['std_claim_amount'] = member_features['std_claim_amount'].fillna(0)

print(f"Feature matrix shape: {member_features.shape}")
print(f"\nFeatures created:")
for col in member_features.columns[1:]:
    print(f"  {col}")
print(f"\nSample data:")
member_features.head()

## 3. Distribution & Outlier Visualization

In [ ]:
fig, axes = plt.subplots(2, 3
    , figsize=(16, 10))

plot_features = ['avg_claim_amount'
    , 'total_claims'
    , 'max_claim_amount'
    , 'unique_providers'
    , 'weekend_ratio'
    , 'denied_ratio']
                 
plot_titles = ['Avg Claim Amount'
    , 'Total Claims'
    , 'Max Single Claim',
    'Unique Providers'
    , 'Weekend Claims Ratio'
    , 'Denied Claims Ratio']

for idx, (feat, title) in enumerate(zip(plot_features, plot_titles)):
    ax = axes[idx // 3][idx % 3]
    ax.hist(member_features[feat], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(member_features[feat].mean(), color='red', linestyle='--', label='Mean')
    ax.axvline(member_features[feat].quantile(0.95), color='orange', linestyle='--', label='95th pctl')
    ax.set_title(f'Distribution: {title}')
    ax.set_xlabel(feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nOutlier Summary (values > 95th percentile):")
for feat, title in zip(plot_features, plot_titles):
    threshold = member_features[feat].quantile(0.95)
    n_outliers = (member_features[feat] > threshold).sum()
    print(f"  {title:<25s}: {n_outliers} members above {threshold:.2f}")

## 4. Train Isolation Forest Model

In [ ]:
from sklearn.ensemble 
import IsolationForest
from sklearn.preprocessing 
import StandardScaler

feature_cols = ['total_claims', 'total_amount', 'avg_claim_amount', 'max_claim_amount',
                'std_claim_amount', 'unique_providers', 'claim_frequency_per_month',
                'weekend_ratio', 'denied_ratio', 'out_of_network_ratio', 'amount_per_provider']

X = member_features[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_scaled)

member_features['anomaly_score'] = iso_forest.decision_function(X_scaled)
member_features['anomaly_label'] = iso_forest.predict(X_scaled)
member_features['is_fraud'] = (member_features['anomaly_label'] == -1).astype(int)

print(f"Model trained successfully!")
print(f"\nResults:")
print(f"  Total members analyzed: {len(member_features)}")
print(f"  Flagged as suspicious:  {member_features['is_fraud'].sum()} ({member_features['is_fraud'].mean()*100:.1f}%)")
print(f"  Normal members:         {(member_features['is_fraud']==0).sum()}")
print(f"\nAnomaly Score Distribution:")
print(f"  Mean:  {member_features['anomaly_score'].mean():.4f}")
print(f"  Min:   {member_features['anomaly_score'].min():.4f} (most anomalous)")
print(f"  Max:   {member_features['anomaly_score'].max():.4f} (most normal)")

## 5. Evaluate & Analyze Anomalies

In [ ]:
fraud_members = member_features[member_features['is_fraud'] == 1].sort_values('anomaly_score')
normal_members = member_features[member_features['is_fraud'] == 0]

print("="*70)
print("  FRAUD vs NORMAL MEMBER COMPARISON")
print("="*70)
print(f"{'Metric':<30} {'Normal':>12} {'Suspicious':>12} {'Ratio':>8}")
print("-"*70)

for col in feature_cols:
    normal_avg = normal_members[col].mean()
    fraud_avg = fraud_members[col].mean()
    ratio = fraud_avg / normal_avg if normal_avg > 0 else 0
    print(f"{col:<30} {normal_avg:>12.2f} {fraud_avg:>12.2f} {ratio:>7.1f}x")

print("\n" + "="*70)
print("\nTop 10 Most Suspicious Members:")
print("-"*70)
for _, row in fraud_members.head(10).iterrows():
    print(f"  {row['MEMBER_ID']}: Score={row['anomaly_score']:.4f}, "
          f"Claims={row['total_claims']:.0f}, "
          f"Avg=${row['avg_claim_amount']:,.0f}, "
          f"Max=${row['max_claim_amount']:,.0f}, "
          f"Providers={row['unique_providers']:.0f}")

## 6. Visualize Anomalies

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['steelblue' if x == 0 else 'crimson' for x in member_features['is_fraud']]
axes[0][0].scatter(member_features['total_claims'], member_features['avg_claim_amount'],
                   c=colors, alpha=0.5, s=20)
axes[0][0].set_xlabel('Total Claims')
axes[0][0].set_ylabel('Avg Claim Amount ($)')
axes[0][0].set_title('Total Claims vs Avg Amount (Red = Suspicious)')

axes[0][1].scatter(member_features['unique_providers'], member_features['total_amount'],
                   c=colors, alpha=0.5, s=20)
axes[0][1].set_xlabel('Unique Providers')
axes[0][1].set_ylabel('Total Amount ($)')
axes[0][1].set_title('Provider Diversity vs Total Spend')

axes[1][0].hist(member_features[member_features['is_fraud']==0]['anomaly_score'], 
               bins=50, alpha=0.7, color='steelblue', label='Normal')
axes[1][0].hist(member_features[member_features['is_fraud']==1]['anomaly_score'], 
               bins=20, alpha=0.7, color='crimson', label='Suspicious')
axes[1][0].set_xlabel('Anomaly Score')
axes[1][0].set_ylabel('Count')
axes[1][0].set_title('Anomaly Score Distribution')
axes[1][0].legend()

feat_importance = []
for col in feature_cols:
    normal_std = normal_members[col].std()
    if normal_std > 0:
        diff = abs(fraud_members[col].mean() - normal_members[col].mean()) / normal_std
    else:
        diff = 0
    feat_importance.append((col, diff))

feat_importance.sort(key=lambda x: x[1], reverse=True)
feat_names = [f[0] for f in feat_importance]
feat_values = [f[1] for f in feat_importance]

axes[1][1].barh(feat_names, feat_values, color='darkorange')
axes[1][1].set_xlabel('Standardized Difference (Normal vs Fraud)')
axes[1][1].set_title('Feature Importance for Fraud Detection')

plt.tight_layout()
plt.show()

## 7. Register Model in Snowflake ML Registry

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model.type_hints import TargetPlatform
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE DATABASE HEALTHCARE').collect()
session.sql('USE SCHEMA ENROLLMENT').collect()

reg = Registry(session=session, database_name='HEALTHCARE', schema_name='ENROLLMENT')

try:
    reg.delete_model('CLAIMS_FRAUD_DETECTOR')
    print('Deleted old model version...')
except:
    pass

sample_input = pd.DataFrame(X_scaled[:10], columns=feature_cols)

mv = reg.log_model(
    model=iso_forest,
    model_name='CLAIMS_FRAUD_DETECTOR',
    version_name='V1',
    target_platforms=[TargetPlatform.SNOWPARK_CONTAINER_SERVICES],
    sample_input_data=sample_input,
    metrics={
        'contamination': 0.05,
        'n_estimators': 200,
        'total_members_analyzed': len(member_features),
        'fraud_flagged_count': int(member_features['is_fraud'].sum()),
        'fraud_flagged_pct': round(member_features['is_fraud'].mean() * 100, 2)
    },
    comment='Isolation Forest anomaly detection for healthcare claims fraud - 11 features'
)

print('Model registered successfully!')
print(f'Model: HEALTHCARE.ENROLLMENT.CLAIMS_FRAUD_DETECTOR')
print(f'Version: V1')
print(f'Fraud detection rate: {member_features["is_fraud"].mean()*100:.1f}%')

## 8. Test Inference

In [ ]:
test_members = member_features[feature_cols].head(10)
test_scaled = pd.DataFrame(scaler.transform(test_members), columns=feature_cols)

test_df = session.create_dataframe(test_scaled)
result = mv.run(test_df, function_name='predict')
result.show()

print('\nModel inference working!')
print('\nFraud Detection Pipeline Complete:')
print('  1. Claims data created from member enrollment')
print('  2. 11 behavioral features engineered')
print('  3. Isolation Forest trained (unsupervised)')
print('  4. 5% most anomalous members flagged')
print('  5. Model registered in Snowflake ML Registry')